# Ingest bronze stream

In [1]:
import os
from datetime import datetime, timezone

import pyspark.sql.functions as F
from confluent_kafka.schema_registry import Schema, SchemaRegistryClient
from delta.tables import DeltaTable
from dotenv import load_dotenv
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.avro.functions import from_avro

load_dotenv('../../../../.env_file/infra.env'), load_dotenv('../../../../.env_file/app.env')

(True, True)

In [2]:
SCHEMA_STORE = 'warehouse'
CHECKPOINT = '/home/nblong/Dropbox/Project/ops/ml-ops/spark/checkpoint'

spark = (
    SparkSession.builder
    .appName('ingest data to bronze zone stream')

    # read/write S3
    .config("spark.hadoop.fs.s3a.access.key", os.getenv('AWS_ACCESS_KEY_ID'))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv('AWS_SECRET_ACCESS_KEY'))
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")

    # dependencies for streaming
    .config(
        'spark.jars.packages',
        'org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,'
        'org.apache.spark:spark-avro_2.13:4.1.1,'
        'org.postgresql:postgresql:42.7.11,'
        'org.apache.hadoop:hadoop-aws:3.4.2,'
        'software.amazon.awssdk:bundle:2.29.52,'
        'io.delta:delta-spark_4.1_2.13:4.3.0,'
    )
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config('spark.sql.streaming.checkpointLocation', CHECKPOINT)
    .config('spark.sql.streaming.forceDeleteTempCheckpointLocation', 'true')

    # Hive metastore
    .config("spark.hadoop.hive.metastore.uris", "thrift://localhost:9084")
    .config("spark.sql.catalogImplementation", "hive")
    # create schema using spark (instead of using trino)
    .config("spark.sql.warehouse.dir", os.path.join(os.getenv('S3_URI'), SCHEMA_STORE))
    .enableHiveSupport()

    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/28 10:34:09 WARN Utils: Your hostname, debian-nblong, resolves to a loopback address: 127.0.1.1; using 192.168.36.29 instead (on interface wlp0s20f3)
26/09/28 10:34:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/nblong/Dropbox/Project/ops/ml-ops/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/nblong/.ivy2.5.2/cache
The jars for the packages stored in: /home/nblong/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.spark#spark-avro_2.13 added as a dependency
org.postgresql#postgresql added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#

- Đọc từ Postgres. Tạm thời lưu 1 bộ data tĩnh để xử lý nháp

In [3]:
bootstrap_servers = 'localhost:9092'
topic = 'source.public.nyc_taxi_stream'

In [4]:
# get stream data
df_stream = (
    spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', bootstrap_servers)
    .option('subscribe', topic)
    .option('startingOffsets', 'latest') # `earliest` hoặc `latest`
    .option('failOnDataLoss', 'false')
    .load()
)

In [ ]:
# get static data from stream
(
    df_stream
    .writeStream
    .queryName("tmp_table")
    .outputMode('append')
    .format('memory') # to save data to memory (so we can examine them)
    .option("truncate", "false")
    .trigger(processingTime='2 seconds')
    .start()
).awaitTermination(5*60)

df_static = spark.sql('select * from tmp_table')
df_static.show()

26/09/23 09:25:49 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


+----+--------------------+--------------------+---------+------+--------------------+-------------+
| key|               value|               topic|partition|offset|           timestamp|timestampType|
+----+--------------------+--------------------+---------+------+--------------------+-------------+
|NULL|[00 00 00 00 03 0...|source.public.nyc...|        0|     0|2026-09-23 09:25:...|            0|
|NULL|[00 00 00 00 03 0...|source.public.nyc...|        0|     1|2026-09-23 09:25:...|            0|
|NULL|[00 00 00 00 03 0...|source.public.nyc...|        0|     2|2026-09-23 09:25:...|            0|
|NULL|[00 00 00 00 03 0...|source.public.nyc...|        0|     3|2026-09-23 09:25:...|            0|
|NULL|[00 00 00 00 03 0...|source.public.nyc...|        0|     4|2026-09-23 09:25:...|            0|
|NULL|[00 00 00 00 03 0...|source.public.nyc...|        0|     5|2026-09-23 09:25:...|            0|
|NULL|[00 00 00 00 03 0...|source.public.nyc...|        0|     6|2026-09-23 09:25:...|     

In [14]:
# đoạn code này đọc toàn bộ kafka (không đợi streaming event vì tốn thời gian)

df_static = (
    spark.read
    .format('kafka')
    .option('kafka.bootstrap.servers', bootstrap_servers)
    .option('subscribe', topic)
    .option('startingOffsets', 'earliest')
    .option('failOnDataLoss', 'false')
    .load()
)

In [15]:
df_static.count()

27621

In [5]:
def get_schema_from_registry(subject:str, schema_registry_url:str="http://localhost:8081")->str:
    client = SchemaRegistryClient({"url": schema_registry_url})
    return client.get_latest_version(subject).to_dict()

def parse_cdc_event(df:DataFrame, topic:str, schema_registry_url:str='http://localhost:8081'):
    # Fetch value schema from Schema Registry
    value_subject = f'{topic}-value'

    try:
        value_schema = get_schema_from_registry(value_subject, schema_registry_url)['schema']

        # Deserialize Avro data
        # Skip first 5 bytes (Confluent wire format: magic byte + 4-byte schema ID)
        decoded_df = df.select(
            from_avro(
                F.expr("substring(value, 6)"),  # bỏ 5 bytes đầu (1 magic + 4 schema_id)
                value_schema
            ).alias("payload"),
            F.col("timestamp").alias("kafka_ts"),
            F.col("partition"),
            F.col("offset"),
        )

        # Extract fields - after ExtractNewRecordState transform, Debezium adds __op, __table
        # Use recorded_at as event_time (it's already a ZonedTimestamp string from Debezium)
        return decoded_df.select(
            F.col('payload.*')
        )

    except Exception as e:
        print(f'Error parsing Avro data: {e}')
        raise

- Lấy schema đã được Debezium đăng ký trên schema-registry về để decode avro

In [ ]:
value_schema = get_schema_from_registry(f'{topic}-value')['schema']
value_schema

'{"type":"record","name":"Value","namespace":"source.public.nyc_taxi_stream","fields":[{"name":"vendor_id","type":["null","string"],"default":null},{"name":"pickup_datetime","type":["null",{"type":"long","connect.version":1,"connect.name":"io.debezium.time.MicroTimestamp"}],"default":null},{"name":"dropoff_datetime","type":["null",{"type":"long","connect.version":1,"connect.name":"io.debezium.time.MicroTimestamp"}],"default":null},{"name":"passenger_count","type":["null","double"],"default":null},{"name":"trip_distance","type":["null","double"],"default":null},{"name":"rate_code_id","type":["null","double"],"default":null},{"name":"store_and_fwd_flag","type":["null","string"],"default":null},{"name":"pickup_longitude","type":["null","double"],"default":null},{"name":"pickup_latitude","type":["null","double"],"default":null},{"name":"dropoff_longitude","type":["null","double"],"default":null},{"name":"dropoff_latitude","type":["null","double"],"default":null},{"name":"pu_location_id","t

- Parse msg theo schema lấy được

In [ ]:
parsed_df = parse_cdc_event(df_static, topic=topic)
parsed_df.show(10)

26/09/23 11:12:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+----------------+----------------+---------------+-------------+------------+------------------+----------------+---------------+-----------------+----------------+--------------+--------------+------------+-----------+-----+-------+---------+---------------------+----------+------------+------------+--------------------+-----------+------------------+--------------------+-----------------------+---------+----+---------------+
|vendor_id| pickup_datetime|dropoff_datetime|passenger_count|trip_distance|rate_code_id|store_and_fwd_flag|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|pu_location_id|do_location_id|payment_type|fare_amount|extra|mta_tax|surcharge|improvement_surcharge|tip_amount|tolls_amount|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|         source_file|source_created_datetime|__deleted|__op|        __table|
+---------+----------------+----------------+---------------+-------------+------------+------------------+---------

In [8]:
parsed_df.dtypes

[('vendor_id', 'string'),
 ('pickup_datetime', 'bigint'),
 ('dropoff_datetime', 'bigint'),
 ('passenger_count', 'double'),
 ('trip_distance', 'double'),
 ('rate_code_id', 'double'),
 ('store_and_fwd_flag', 'string'),
 ('pickup_longitude', 'double'),
 ('pickup_latitude', 'double'),
 ('dropoff_longitude', 'double'),
 ('dropoff_latitude', 'double'),
 ('pu_location_id', 'string'),
 ('do_location_id', 'string'),
 ('payment_type', 'string'),
 ('fare_amount', 'double'),
 ('extra', 'double'),
 ('mta_tax', 'double'),
 ('surcharge', 'double'),
 ('improvement_surcharge', 'double'),
 ('tip_amount', 'double'),
 ('tolls_amount', 'double'),
 ('total_amount', 'double'),
 ('congestion_surcharge', 'double'),
 ('airport_fee', 'double'),
 ('cbd_congestion_fee', 'double'),
 ('source_file', 'string'),
 ('source_created_datetime', 'string'),
 ('__deleted', 'string'),
 ('__op', 'string'),
 ('__table', 'string')]

In [6]:
# chạy stream
parsed_df = parse_cdc_event(df_stream, topic=topic)

- Cần phải convert 2 trường event time sang kiểu Timestamp để phù hợp với dữ liệu ghi trong Bronze hiện tại
- Lý do tôi xử lý ở Bronze (bình thường chỉ chứa dữ liệu thô) là vì trong thực tế, Bronze zone chỉ nên là **1 location**. Batch được trigger theo ngày, còn stream hoạt động liên tục. Do đó, dữ liệu trong 1 location nên là đồng nhất
- Ở đây dùng location cho stream khác với batch (mục tiêu chỉ là để mô phỏng stream processing)

In [7]:
df_bronze = (
    parsed_df
    .drop('__deleted', '__op', '__table')
    .withColumn('pickup_datetime', (F.col('pickup_datetime')/1_000_000).cast('timestamp'))
    .withColumn('dropoff_datetime', (F.col('dropoff_datetime')/1_000_000).cast('timestamp'))
)

# # nếu `parsed_df` là stream df thì dòng này sẽ báo lỗi
# df_bronze.show(truncate=False)

In [8]:
df_bronze.dtypes

[('vendor_id', 'string'),
 ('pickup_datetime', 'timestamp'),
 ('dropoff_datetime', 'timestamp'),
 ('passenger_count', 'double'),
 ('trip_distance', 'double'),
 ('rate_code_id', 'double'),
 ('store_and_fwd_flag', 'string'),
 ('pickup_longitude', 'double'),
 ('pickup_latitude', 'double'),
 ('dropoff_longitude', 'double'),
 ('dropoff_latitude', 'double'),
 ('pu_location_id', 'string'),
 ('do_location_id', 'string'),
 ('payment_type', 'string'),
 ('fare_amount', 'double'),
 ('extra', 'double'),
 ('mta_tax', 'double'),
 ('surcharge', 'double'),
 ('improvement_surcharge', 'double'),
 ('tip_amount', 'double'),
 ('tolls_amount', 'double'),
 ('total_amount', 'double'),
 ('congestion_surcharge', 'double'),
 ('airport_fee', 'double'),
 ('cbd_congestion_fee', 'double'),
 ('source_file', 'string'),
 ('source_created_datetime', 'string')]

- Tạo `record_key`

In [9]:
df_bronze = df_bronze.withColumn(
    'record_key',
    F.sha2(F.concat_ws(
        "||",
        *sorted(set(df_bronze.columns).difference({'source_file', 'source_created_datetime'}))
    ), 256)
)

- Ghi ra Delta Lake (S3). Tôi muốn ghi với config sau
    - schema: `bronze`
    - table: `lakehouse.bronze.nyc_taxi_stream`
    - path: `s3://gestore-stuff/nyc_taxi/bronze_stream`

In [10]:
SCHEMA = "bronze"
S3_BRONZE_PATH = os.path.join(os.getenv('S3_URI'), f'{SCHEMA}_stream')
TABLE  = "nyc_taxi_stream"
SCHEMA_STORE = 'warehouse'

In [11]:
# partition ingested time by minute
bronze_created_datetime = datetime.now(timezone.utc)
month_partition_df = (
    df_bronze
    .withColumn('bronze_created_datetime', F.lit(bronze_created_datetime).cast('timestamp'))
    .withColumn('year', F.substring('source_file', 17, 4).cast('int'))
    .withColumn('month', F.substring('source_file', 22, 2).cast('int'))
)

# create schema in lake house
spark.sql(f'create database if not exists {SCHEMA}')
print(f'\nCreated {SCHEMA}\n')

# get current version
table_name = f'{SCHEMA}.{TABLE}'
if spark.catalog.tableExists(table_name):
    delta_table = DeltaTable.forName(spark, table_name)
    current_version = (
        delta_table.history(limit=1)
        .select('version').collect()[0][0]
    )
else:
    current_version = -1

# save to lake house
query = (
    month_partition_df
    .repartition("year", "month") # prevent small files problem
    .writeStream
    .format('delta')
    .option("delta.enableDeletionVectors", "true") # have deletion vectors on
    .partitionBy("year", "month")
    .option("checkpointLocation", CHECKPOINT)
    .option('path', S3_BRONZE_PATH)
    .trigger(processingTime='1 minute')
    .toTable(tableName=f'{SCHEMA}.{TABLE}')
)

query.awaitTermination()


Created bronze



26/09/28 10:48:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/09/28 10:48:43 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`bronze`.`nyc_taxi_stream` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/09/28 10:48:43 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/09/28 10:48:49 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/09/28 10:48:49 WARN KafkaOffsetReaderAdmin: Found incorrect offsets in some partitions (partition, previous offset, fetched offset): List((source.public.nyc_taxi_stream-0,2821,0))
26/09/28 10:48:49 WARN KafkaOffsetReaderAdmin: Re

KeyboardInterrupt: 

In [12]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, table_name)
delta_table.optimize().executeCompaction()

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [13]:
delta_table.optimize().executeZOrderBy('record_key').show()

+--------------------+--------------------+
|                path|             metrics|
+--------------------+--------------------+
|s3a://gestore-stu...|{1, 1, {2695987, ...|
+--------------------+--------------------+



In [14]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
delta_table.vacuum(0).show()

Deleted 7 files and directories in a total of 3 directories.
++
||
++
++



In [16]:
delta_table.history().select('version', 'timestamp', 'operation').show(truncate=True)

+-------+-------------------+----------------+
|version|          timestamp|       operation|
+-------+-------------------+----------------+
|      9|2026-09-28 11:00:55|        OPTIMIZE|
|      8|2026-09-28 11:00:15|        OPTIMIZE|
|      7|2026-09-28 10:56:03|STREAMING UPDATE|
|      6|2026-09-28 10:55:08|STREAMING UPDATE|
|      5|2026-09-28 10:54:10|STREAMING UPDATE|
|      4|2026-09-28 10:53:09|STREAMING UPDATE|
|      3|2026-09-28 10:52:08|STREAMING UPDATE|
|      2|2026-09-28 10:51:10|STREAMING UPDATE|
|      1|2026-09-28 10:48:57|STREAMING UPDATE|
|      0|2026-09-28 10:48:39|    CREATE TABLE|
+-------+-------------------+----------------+



In [36]:
delta_table.vacuum(0).show()

Deleted 1 files and directories in a total of 3 directories.
++
||
++
++



In [24]:
spark.catalog.tableExists('bronze.nyc_taxi_stream')

False

26/09/28 19:23:13 WARN TransportChannelHandler: Exception in connection from 192.168.36.29/192.168.36.29:43055
io.netty.channel.unix.Errors$NativeIoException: recvAddress(..) failed with error(-110): Connection timed out
26/09/28 20:00:18 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 914424 ms exceeds timeout 120000 ms
26/09/28 20:00:18 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/28 20:00:26 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.